# Step 0: Training Analysis
## Part A: Transform Calibration + Part B: Match Classifier Training

One-time analysis over all 6 training subjects.
Outputs:
- `data/coreg_transform_template.json`
- `data/match_classifier.pkl`

In [ ]:
import sys, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path('.')))

from coreg_data_loading import (
    find_coreg_dirs, parse_coreg_dir_name, load_filepaths,
    load_czstack_centroids, load_hcr_centroids, load_hcr_scales,
    load_hcr_metrics, load_spot_counts, load_landmarks, load_subject_data
)
from coreg_alignment import (
    rotation_matrix_euler, euler_from_rotation_matrix, CZ_RESOLUTION_UM
)
from coreg_verification import (
    extract_candidate_features, train_match_classifier, predict_match_probability
)

%load_ext autoreload
%autoreload 2

DATA_DIR = Path('/root/capsule/data')
SAVE_TEMPLATE = DATA_DIR / 'coreg_transform_template.json'
SAVE_CLASSIFIER = DATA_DIR / 'match_classifier.pkl'
GFP_THRESHOLD = 5


## Part A: Rigid Transform Calibration
Fit SVD rigid transform per subject, decompose to Euler angles, measure Z-scale ratio.

In [ ]:
def fit_rigid_svd(src_pts, dst_pts):
    """Fit rotation R and translation t such that dst ~ R @ src + t (Procrustes)."""
    src_c = src_pts.mean(axis=0)
    dst_c = dst_pts.mean(axis=0)
    H = (src_pts - src_c).T @ (dst_pts - dst_c)
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1] *= -1
        R = Vt.T @ U.T
    t = dst_c - R @ src_c
    return R, t


coreg_dirs = find_coreg_dirs(DATA_DIR)
print(f'Found {len(coreg_dirs)} coreg directories:')
for d in coreg_dirs:
    print(' ', d.name)


In [ ]:
from scipy.spatial import cKDTree

def _pick_coreg_table(coreg_dir):
    """Prefer the final coreg table (no _iter suffix) over iteration tables."""
    all_csv = sorted(coreg_dir.glob('*coreg_table*.csv'))
    non_iter = [f for f in all_csv if 'iter' not in f.stem.split('coreg_table')[-1]]
    return non_iter[-1] if non_iter else (all_csv[-1] if all_csv else None)


calibration_results = []

for coreg_dir in coreg_dirs:
    subject_id, czstack_date = parse_coreg_dir_name(coreg_dir)
    print(f'\n=== {subject_id} ===')

    # --- Load final QCed landmarks ---
    lm_files = sorted(coreg_dir.glob('*landmarks*qced*.csv'))
    if not lm_files:
        lm_files = [f for f in sorted(coreg_dir.glob('*landmarks*.csv'))
                    if 'initial' not in f.name]
    if not lm_files:
        print('  No landmarks found, skipping'); continue
    lm_path = lm_files[-1]
    print(f'  Landmarks: {lm_path.name}')
    try:
        lm_df = load_landmarks(lm_path)
    except Exception as e:
        print(f'  Error loading landmarks: {e}'); continue
    active = lm_df[(lm_df['active'] == True) & lm_df['ids'].str.startswith('cz')].copy()
    if len(active) < 5:
        print(f'  Only {len(active)} active landmarks, skipping'); continue

    # --- Load scales and filepaths ---
    try:
        fps = load_filepaths(coreg_dir, subject_id, czstack_date)
        hcr_scales = load_hcr_scales(fps['fused_json_file'])
    except Exception as e:
        print(f'  Error loading scales: {e}'); continue

    cz_res = CZ_RESOLUTION_UM  # (z,y,x) µm/px
    hcr_res = np.array([hcr_scales['scale_z'], hcr_scales['scale_y'], hcr_scales['scale_x']])

    # CZ µm and HCR µm from active landmarks
    src = np.stack([
        active['czstack_z'].values * cz_res[0],
        active['czstack_y'].values * cz_res[1],
        active['czstack_x'].values * cz_res[2],
    ], axis=1)
    dst = np.stack([
        active['hcr_z'].values * hcr_res[0],
        active['hcr_y'].values * hcr_res[1],
        active['hcr_x'].values * hcr_res[2],
    ], axis=1)

    # --- Rigid SVD fit: R, t ---
    R, t = fit_rigid_svd(src, dst)
    tz, tx, ty = euler_from_rotation_matrix(R)

    # --- Scale ratios (extent-based, per axis) ---
    def extent_ratio(a, b, dim):
        ea = a[:, dim].max() - a[:, dim].min()
        return (b[:, dim].max() - b[:, dim].min()) / max(1e-3, ea)
    z_scale = extent_ratio(src, dst, 0)
    y_scale = extent_ratio(src, dst, 1)
    x_scale = extent_ratio(src, dst, 2)

    # --- Load ALL HCR cells for volume bounds ---
    try:
        hcr_all_df = load_hcr_centroids(fps['hcr_centroid_path'])
        hcr_all_um = np.stack([
            hcr_all_df['hcr_z'].values * hcr_res[0],
            hcr_all_df['hcr_y'].values * hcr_res[1],
            hcr_all_df['hcr_x'].values * hcr_res[2],
        ], axis=1)
    except Exception as e:
        print(f'  Cannot load HCR cells: {e}'); continue

    hcr_vol_min   = hcr_all_um.min(axis=0)
    hcr_vol_max   = hcr_all_um.max(axis=0)
    hcr_vol_size  = hcr_vol_max - hcr_vol_min
    hcr_vol_center = (hcr_vol_min + hcr_vol_max) / 2

    # --- Load coreg table → matched HCR positions ---
    ct_path = _pick_coreg_table(coreg_dir)
    if ct_path is None:
        print('  No coreg table found, skipping'); continue
    print(f'  Coreg table: {ct_path.name}')
    coreg_table = pd.read_csv(ct_path)
    if 'czstack_id' in coreg_table.columns:
        coreg_table = coreg_table.rename(columns={'czstack_id': 'czstack_cell_id', 'hcr_id': 'hcr_cell_id'})
    elif 'czstack_cell_id' not in coreg_table.columns:
        coreg_table.columns = ['czstack_cell_id', 'hcr_cell_id']
    n_manual_matches = len(coreg_table)

    hcr_id_arr    = hcr_all_df['hcr_cell_id'].values
    hcr_id_to_idx = {int(hid): i for i, hid in enumerate(hcr_id_arr)}
    matched_rows  = [hcr_id_to_idx[int(h)] for h in coreg_table['hcr_cell_id']
                     if int(h) in hcr_id_to_idx]
    matched_hcr_um = hcr_all_um[matched_rows]

    # --- Blank margins: matched region vs full HCR cell extent ---
    # Z
    margin_z_min_um   = matched_hcr_um[:, 0].min() - hcr_vol_min[0]
    margin_z_max_um   = hcr_vol_max[0] - matched_hcr_um[:, 0].max()
    margin_z_min_frac = margin_z_min_um  / max(1e-3, hcr_vol_size[0])
    margin_z_max_frac = margin_z_max_um  / max(1e-3, hcr_vol_size[0])
    span_z_frac       = (matched_hcr_um[:, 0].max() - matched_hcr_um[:, 0].min()) / max(1e-3, hcr_vol_size[0])
    # Y
    margin_y_min_um   = matched_hcr_um[:, 1].min() - hcr_vol_min[1]
    margin_y_max_um   = hcr_vol_max[1] - matched_hcr_um[:, 1].max()
    margin_y_min_frac = margin_y_min_um  / max(1e-3, hcr_vol_size[1])
    margin_y_max_frac = margin_y_max_um  / max(1e-3, hcr_vol_size[1])
    span_y_frac       = (matched_hcr_um[:, 1].max() - matched_hcr_um[:, 1].min()) / max(1e-3, hcr_vol_size[1])
    # X
    margin_x_min_um   = matched_hcr_um[:, 2].min() - hcr_vol_min[2]
    margin_x_max_um   = hcr_vol_max[2] - matched_hcr_um[:, 2].max()
    margin_x_min_frac = margin_x_min_um  / max(1e-3, hcr_vol_size[2])
    margin_x_max_frac = margin_x_max_um  / max(1e-3, hcr_vol_size[2])
    span_x_frac       = (matched_hcr_um[:, 2].max() - matched_hcr_um[:, 2].min()) / max(1e-3, hcr_vol_size[2])

    # --- CZ projected centroid position in HCR volume ---
    cz_centroid_um      = src.mean(axis=0)
    cz_proj_centroid_um = R @ cz_centroid_um + t         # in HCR µm
    cz_center_frac      = (cz_proj_centroid_um - hcr_vol_min) / hcr_vol_size  # [0,1]³
    t_from_hcr_center   = cz_proj_centroid_um - hcr_vol_center  # signed offset (µm)

    # --- NN distances among matched HCR cells (calibrates seed threshold) ---
    if len(matched_hcr_um) > 1:
        tree_m = cKDTree(matched_hcr_um)
        nn_dists, _ = tree_m.query(matched_hcr_um, k=2)
        nn_dists    = nn_dists[:, 1]  # exclude self
        nn_dist_p50 = float(np.median(nn_dists))
        nn_dist_p90 = float(np.percentile(nn_dists, 90))
    else:
        nn_dist_p50 = nn_dist_p90 = np.nan

    # --- GFP+ density in matched subregion bounding box ---
    try:
        spot_path = fps.get('spot_488_counts_path')
        if spot_path and Path(spot_path).exists():
            sc = pd.read_csv(spot_path)
        else:
            sc_paths  = sorted(coreg_dir.glob('*spot_488*'))
            sc_paths += sorted((Path('/root/capsule/scratch') /
                                f'{subject_id}_{czstack_date}_coreg_cpsam').glob('*spot_488*'))
            sc = pd.read_csv(sc_paths[-1]) if sc_paths else None
        if sc is not None:
            count_col = next((c for c in sc.columns if 'count' in c.lower()), None)
            id_col    = next((c for c in sc.columns if 'hcr' in c.lower() or c.lower() == 'id'), sc.columns[0])
            gfp_ids   = set(sc.loc[sc[count_col] >= GFP_THRESHOLD, id_col].values) if count_col \
                        else set(sc[id_col].values)
            m_z0, m_z1 = matched_hcr_um[:, 0].min(), matched_hcr_um[:, 0].max()
            m_y0, m_y1 = matched_hcr_um[:, 1].min(), matched_hcr_um[:, 1].max()
            m_x0, m_x1 = matched_hcr_um[:, 2].min(), matched_hcr_um[:, 2].max()
            in_box = (
                (hcr_all_um[:, 0] >= m_z0) & (hcr_all_um[:, 0] <= m_z1) &
                (hcr_all_um[:, 1] >= m_y0) & (hcr_all_um[:, 1] <= m_y1) &
                (hcr_all_um[:, 2] >= m_x0) & (hcr_all_um[:, 2] <= m_x1)
            )
            hcr_in_box_ids      = set(hcr_id_arr[in_box].tolist())
            n_gfp_in_box        = len(gfp_ids & hcr_in_box_ids)
            box_vol_um3         = (m_z1-m_z0) * (m_y1-m_y0) * (m_x1-m_x0)
            gfp_density_per_mm3 = n_gfp_in_box / max(1e-9, box_vol_um3) * 1e9
        else:
            n_gfp_in_box = gfp_density_per_mm3 = np.nan
    except Exception as e:
        print(f'  GFP density error: {e}')
        n_gfp_in_box = gfp_density_per_mm3 = np.nan

    # --- Match coverage ---
    try:
        czstack_df = load_czstack_centroids(fps['czstack_centroid_path'])
        n_cz_total = len(czstack_df)
    except Exception:
        n_cz_total = np.nan
    coverage = n_manual_matches / max(1, n_cz_total) if not np.isnan(n_cz_total) else np.nan

    # --- Print ---
    print(f'  Euler:  z={tz:.1f}°  x={tx:.1f}°  y={ty:.1f}°')
    print(f'  Scale:  Z={z_scale:.3f}  Y={y_scale:.3f}  X={x_scale:.3f}')
    print(f'  HCR vol (µm): z={hcr_vol_size[0]:.0f}  y={hcr_vol_size[1]:.0f}  x={hcr_vol_size[2]:.0f}')
    print(f'  CZ center in HCR [frac]: z={cz_center_frac[0]:.2f}  y={cz_center_frac[1]:.2f}  x={cz_center_frac[2]:.2f}')
    print(f'  Offset from HCR center (µm): z={t_from_hcr_center[0]:.0f}  y={t_from_hcr_center[1]:.0f}  x={t_from_hcr_center[2]:.0f}')
    print(f'  Blank Z [frac]: lo={margin_z_min_frac:.2f}  hi={margin_z_max_frac:.2f}  span={span_z_frac:.2f}')
    print(f'  Blank Y [frac]: lo={margin_y_min_frac:.2f}  hi={margin_y_max_frac:.2f}  span={span_y_frac:.2f}')
    print(f'  Blank X [frac]: lo={margin_x_min_frac:.2f}  hi={margin_x_max_frac:.2f}  span={span_x_frac:.2f}')
    print(f'  NN dist (matched HCR): p50={nn_dist_p50:.1f} µm  p90={nn_dist_p90:.1f} µm')
    print(f'  GFP+ density in matched box: {gfp_density_per_mm3:.0f}/mm³  ({n_gfp_in_box} cells)')
    print(f'  Coverage: {n_manual_matches}/{n_cz_total} = {coverage:.1%}')

    calibration_results.append({
        'subject_id':           subject_id,
        'theta_z':              tz,  'theta_x': tx,  'theta_y': ty,
        'z_scale':              z_scale,  'y_scale': y_scale,  'x_scale': x_scale,
        'hcr_vol_z_um':         hcr_vol_size[0],
        'hcr_vol_y_um':         hcr_vol_size[1],
        'hcr_vol_x_um':         hcr_vol_size[2],
        'cz_center_frac_z':     cz_center_frac[0],
        'cz_center_frac_y':     cz_center_frac[1],
        'cz_center_frac_x':     cz_center_frac[2],
        't_from_hcr_center_z':  t_from_hcr_center[0],
        't_from_hcr_center_y':  t_from_hcr_center[1],
        't_from_hcr_center_x':  t_from_hcr_center[2],
        'margin_z_min_um':      margin_z_min_um,
        'margin_z_max_um':      margin_z_max_um,
        'margin_z_min_frac':    margin_z_min_frac,
        'margin_z_max_frac':    margin_z_max_frac,
        'span_z_frac':          span_z_frac,
        'margin_y_min_frac':    margin_y_min_frac,
        'margin_y_max_frac':    margin_y_max_frac,
        'span_y_frac':          span_y_frac,
        'margin_x_min_frac':    margin_x_min_frac,
        'margin_x_max_frac':    margin_x_max_frac,
        'span_x_frac':          span_x_frac,
        'nn_dist_p50_um':       nn_dist_p50,
        'nn_dist_p90_um':       nn_dist_p90,
        'gfp_density_per_mm3':  gfp_density_per_mm3,
        'n_gfp_in_box':         n_gfp_in_box,
        'n_manual_matches':     n_manual_matches,
        'n_cz_total':           n_cz_total,
        'coverage':             coverage,
        'n_landmarks':          len(active),
    })

calib_df = pd.DataFrame(calibration_results)
print('\n=== Summary ===')
print(calib_df[['subject_id', 'theta_z', 'theta_x', 'theta_y',
                 'z_scale', 'y_scale', 'x_scale', 'coverage',
                 'cz_center_frac_z', 'cz_center_frac_y', 'cz_center_frac_x',
                 'span_z_frac', 'nn_dist_p50_um']].to_string())

In [ ]:
# Build search template from calibration results
margin_deg = 10  # degrees padding beyond observed range

if len(calib_df) == 0:
    print('WARNING: No calibration data — using defaults')
    template = {
        'z_rotation_range_deg': [170, 190],
        'pitch_range_deg': [-185, -170],
        'roll_range_deg': [-10, 10],
        'z_scale_mean': 2.8, 'z_scale_std': 0.3,
        'xy_scale_mean': 1.0, 'xy_scale_std': 0.1,
        'cz_center_frac_z_mean': 0.5, 'cz_center_frac_z_std': 0.15,
        'cz_center_frac_y_mean': 0.5, 'cz_center_frac_y_std': 0.15,
        'cz_center_frac_x_mean': 0.5, 'cz_center_frac_x_std': 0.15,
        'margin_z_min_frac_mean': 0.1, 'margin_z_min_frac_std': 0.05,
        'margin_z_max_frac_mean': 0.1, 'margin_z_max_frac_std': 0.05,
        'span_z_frac_mean': 0.3, 'span_z_frac_std': 0.05,
        'margin_y_min_frac_mean': 0.05, 'margin_y_max_frac_mean': 0.05,
        'margin_x_min_frac_mean': 0.05, 'margin_x_max_frac_mean': 0.05,
        'nn_dist_p50_um_mean': 20.0, 'nn_dist_p90_um_mean': 40.0,
        'seed_distance_threshold_um': 25.0,
        'coverage_mean': 0.75, 'coverage_std': 0.05,
    }
else:
    xy_scale_all = pd.concat([calib_df['y_scale'], calib_df['x_scale']])

    # seed distance threshold: 2× the p90 NN distance (generous enough to seed well)
    seed_dist_thresh = float(calib_df['nn_dist_p90_um'].mean() * 2.0)

    template = {
        # --- Rotation search bounds ---
        'z_rotation_range_deg': [
            float(calib_df['theta_z'].min() - margin_deg),
            float(calib_df['theta_z'].max() + margin_deg),
        ],
        'pitch_range_deg': [
            float(calib_df['theta_x'].min() - margin_deg),
            float(calib_df['theta_x'].max() + margin_deg),
        ],
        'roll_range_deg': [
            float(calib_df['theta_y'].min() - margin_deg),
            float(calib_df['theta_y'].max() + margin_deg),
        ],

        # --- Scale ratios ---
        'z_scale_mean':  float(calib_df['z_scale'].mean()),
        'z_scale_std':   float(calib_df['z_scale'].std()),
        'xy_scale_mean': float(xy_scale_all.mean()),
        'xy_scale_std':  float(xy_scale_all.std()),

        # --- HCR volume size (µm) across subjects ---
        'hcr_vol_z_um_mean': float(calib_df['hcr_vol_z_um'].mean()),
        'hcr_vol_y_um_mean': float(calib_df['hcr_vol_y_um'].mean()),
        'hcr_vol_x_um_mean': float(calib_df['hcr_vol_x_um'].mean()),

        # --- CZ centroid fractional position within HCR volume ---
        # Used to constrain translation search: expect CZ center near these fractions
        'cz_center_frac_z_mean': float(calib_df['cz_center_frac_z'].mean()),
        'cz_center_frac_z_std':  float(calib_df['cz_center_frac_z'].std()),
        'cz_center_frac_y_mean': float(calib_df['cz_center_frac_y'].mean()),
        'cz_center_frac_y_std':  float(calib_df['cz_center_frac_y'].std()),
        'cz_center_frac_x_mean': float(calib_df['cz_center_frac_x'].mean()),
        'cz_center_frac_x_std':  float(calib_df['cz_center_frac_x'].std()),

        # --- Blank margins in Z (as fractions of HCR Z extent) ---
        # How much of HCR Z is above/below the match region
        'margin_z_min_frac_mean': float(calib_df['margin_z_min_frac'].mean()),
        'margin_z_min_frac_std':  float(calib_df['margin_z_min_frac'].std()),
        'margin_z_max_frac_mean': float(calib_df['margin_z_max_frac'].mean()),
        'margin_z_max_frac_std':  float(calib_df['margin_z_max_frac'].std()),
        'span_z_frac_mean':       float(calib_df['span_z_frac'].mean()),
        'span_z_frac_std':        float(calib_df['span_z_frac'].std()),

        # --- Blank margins in Y, X ---
        'margin_y_min_frac_mean': float(calib_df['margin_y_min_frac'].mean()),
        'margin_y_max_frac_mean': float(calib_df['margin_y_max_frac'].mean()),
        'span_y_frac_mean':       float(calib_df['span_y_frac'].mean()),
        'margin_x_min_frac_mean': float(calib_df['margin_x_min_frac'].mean()),
        'margin_x_max_frac_mean': float(calib_df['margin_x_max_frac'].mean()),
        'span_x_frac_mean':       float(calib_df['span_x_frac'].mean()),

        # --- Cell-level statistics ---
        'nn_dist_p50_um_mean':      float(calib_df['nn_dist_p50_um'].mean()),
        'nn_dist_p90_um_mean':      float(calib_df['nn_dist_p90_um'].mean()),
        'gfp_density_per_mm3_mean': float(calib_df['gfp_density_per_mm3'].mean()),

        # --- Seed distance threshold for step_2 ---
        # Set to 2× p90 NN distance: generous enough to seed, tight enough to avoid false matches
        'seed_distance_threshold_um': seed_dist_thresh,

        # --- Match coverage ---
        'coverage_mean': float(calib_df['coverage'].mean()),
        'coverage_std':  float(calib_df['coverage'].std()),

        'n_subjects': int(len(calib_df)),
    }

print('Transform template:')
print(json.dumps(template, indent=2))

with open(SAVE_TEMPLATE, 'w') as f:
    json.dump(template, f, indent=2)
print(f'\nSaved to {SAVE_TEMPLATE}')

In [ ]:
# Compute search bounds from calibration results
if len(calib_df) == 0:
    print('WARNING: No calibration data - using defaults')
    template = {
        'z_rotation_range_deg': [170, 190],
        'pitch_range_deg': [-10, 10],
        'roll_range_deg': [-10, 10],
        'z_scale_mean': 2.8,
        'z_scale_std': 0.3,
    }
else:
    margin = 10  # degrees of margin beyond observed range
    template = {
        'z_rotation_range_deg': [
            float(calib_df['theta_z'].min() - margin),
            float(calib_df['theta_z'].max() + margin),
        ],
        'pitch_range_deg': [
            float(calib_df['theta_x'].min() - margin),
            float(calib_df['theta_x'].max() + margin),
        ],
        'roll_range_deg': [
            float(calib_df['theta_y'].min() - margin),
            float(calib_df['theta_y'].max() + margin),
        ],
        'z_scale_mean': float(calib_df['z_scale'].mean()),
        'z_scale_std': float(calib_df['z_scale'].std()),
        'n_subjects': len(calib_df),
    }

print('Transform template:')
print(json.dumps(template, indent=2))

with open(SAVE_TEMPLATE, 'w') as f:
    json.dump(template, f, indent=2)
print(f'\nSaved to {SAVE_TEMPLATE}')


## Part B: Match Classifier Training
Extract positive/negative examples from all training subjects.

In [ ]:
from scipy.spatial import cKDTree

def extract_training_examples(
    coreg_dir, subject_id, czstack_date,
    n_negatives_per_positive=5,
    negative_radius_um=50.0,
    rng_seed=42,
):
    """Extract labeled positive/negative feature examples for one subject."""
    rng = np.random.default_rng(rng_seed)

    try:
        data = load_subject_data(coreg_dir, subject_id, czstack_date,
                                  gfp_threshold=GFP_THRESHOLD, load_iter_paths=True)
    except Exception as e:
        print(f'  Error loading data: {e}')
        return None, None

    czstack_df = data['czstack_df']
    hcr_df = data['hcr_df']
    hcr_scales = data['hcr_scales']
    spot_counts = data['spot_counts']
    hcr_metrics = data['hcr_metrics']

    # Load coreg table (ground truth)
    coreg_csvs = sorted(coreg_dir.glob(f'*coreg_table.csv'))
    if not coreg_csvs:
        print(f'  No coreg table found')
        return None, None
    coreg_table = pd.read_csv(coreg_csvs[-1])  # czstack_id, hcr_id
    if 'czstack_id' not in coreg_table.columns:
        coreg_table.columns = ['czstack_id', 'hcr_id']
    coreg_table = coreg_table.rename(columns={'czstack_id': 'czstack_cell_id', 'hcr_id': 'hcr_cell_id'})
    print(f'  {len(coreg_table)} ground-truth matches')

    # Load final landmarks for TPS
    lm_files = sorted(coreg_dir.glob('*landmarks*qced*.csv'))
    if not lm_files:
        lm_files = sorted(coreg_dir.glob('*landmarks*.csv'))
    if not lm_files:
        print('  No landmarks found')
        return None, None
    landmarks = load_landmarks(lm_files[-1])

    from coreg_matching import fit_tps, project_czstack_to_hcr
    from coreg_alignment import CZ_RESOLUTION_UM

    try:
        tps = fit_tps(landmarks)
        projected_all = project_czstack_to_hcr(czstack_df, tps)
    except Exception as e:
        print(f'  TPS failed: {e}')
        return None, None

    hcr_res = np.array([hcr_scales['scale_z'], hcr_scales['scale_y'], hcr_scales['scale_x']])

    projected_um_df = pd.DataFrame({
        'czstack_cell_id': czstack_df['czstack_cell_id'].values,
        'projected_z': projected_all[:, 0] * hcr_res[0],
        'projected_y': projected_all[:, 1] * hcr_res[1],
        'projected_x': projected_all[:, 2] * hcr_res[2],
        'czstack_z': czstack_df['czstack_z'].values,
        'czstack_y': czstack_df['czstack_y'].values,
        'czstack_x': czstack_df['czstack_x'].values,
    })

    hcr_centroids_um = pd.DataFrame({
        'hcr_cell_id': hcr_df['hcr_cell_id'].values,
        'hcr_z': hcr_df['hcr_z'].values * hcr_res[0],
        'hcr_y': hcr_df['hcr_y'].values * hcr_res[1],
        'hcr_x': hcr_df['hcr_x'].values * hcr_res[2],
    })

    gfp_hcr = hcr_df[hcr_df['hcr_cell_id'].isin(
        spot_counts[spot_counts['is_gfp'] == True]['hcr_cell_id']
    )].reset_index(drop=True)

    # Build positive examples
    pos_pairs = coreg_table[
        coreg_table['czstack_cell_id'].isin(czstack_df['czstack_cell_id']) &
        coreg_table['hcr_cell_id'].isin(hcr_df['hcr_cell_id'])
    ].copy()

    cz_proj_idx = projected_um_df.set_index('czstack_cell_id')
    hcr_um_idx = hcr_centroids_um.set_index('hcr_cell_id')

    positives = []
    for _, pair in pos_pairs.iterrows():
        cz_id = int(pair['czstack_cell_id'])
        hcr_id = int(pair['hcr_cell_id'])
        try:
            cz_pt = cz_proj_idx.loc[cz_id, ['projected_z', 'projected_y', 'projected_x']].values
            hcr_pt = hcr_um_idx.loc[hcr_id, ['hcr_z', 'hcr_y', 'hcr_x']].values
            dist = float(np.linalg.norm(cz_pt - hcr_pt))
        except KeyError:
            dist = np.nan
        positives.append({'czstack_cell_id': cz_id, 'hcr_cell_id': hcr_id,
                          'distance_um': dist, 'nn_rank': 1, 'is_mutual_nn': True})

    pos_df = pd.DataFrame(positives)
    n_pos = len(pos_df)

    # Negative examples: random non-matching pairs within negative_radius_um
    matched_hcr_set = set(coreg_table['hcr_cell_id'].values)
    matched_cz_set = set(coreg_table['czstack_cell_id'].values)

    proj_coords = projected_um_df[['projected_z', 'projected_y', 'projected_x']].values
    hcr_gfp_um = hcr_centroids_um[hcr_centroids_um['hcr_cell_id'].isin(
        gfp_hcr['hcr_cell_id'])][['hcr_z', 'hcr_y', 'hcr_x']].values
    hcr_gfp_ids = hcr_centroids_um[hcr_centroids_um['hcr_cell_id'].isin(
        gfp_hcr['hcr_cell_id'])]['hcr_cell_id'].values

    tree_hcr = cKDTree(hcr_gfp_um)
    negatives = []
    cz_ids_arr = projected_um_df['czstack_cell_id'].values

    for i, cz_id in enumerate(cz_ids_arr):
        if int(cz_id) in matched_cz_set:
            continue
        # Find HCR cells within radius
        nbrs = tree_hcr.query_ball_point(proj_coords[i], negative_radius_um)
        if not nbrs:
            continue
        # Sample up to n_negatives_per_positive
        sel = rng.choice(nbrs, size=min(n_negatives_per_positive, len(nbrs)), replace=False)
        for j in sel:
            hcr_id = int(hcr_gfp_ids[j])
            if hcr_id in matched_hcr_set:
                continue
            dist = float(np.linalg.norm(proj_coords[i] - hcr_gfp_um[j]))
            negatives.append({'czstack_cell_id': int(cz_id), 'hcr_cell_id': hcr_id,
                               'distance_um': dist, 'nn_rank': 2, 'is_mutual_nn': False})
        if len(negatives) >= n_pos * n_negatives_per_positive:
            break

    neg_df = pd.DataFrame(negatives[:n_pos * n_negatives_per_positive])
    print(f'  {n_pos} positives, {len(neg_df)} negatives')

    all_cands = pd.concat([pos_df, neg_df], ignore_index=True)
    labels = np.array([1] * n_pos + [0] * len(neg_df))

    feats = extract_candidate_features(
        all_cands, projected_um_df, hcr_centroids_um,
        coreg_table.rename(columns={'czstack_id': 'czstack_cell_id', 'hcr_id': 'hcr_cell_id'})
        if 'czstack_id' in coreg_table.columns else coreg_table,
        czstack_vol=None,   # skip NCC for speed in training
        hcr_zarr_path=None,
        czstack_res_um=CZ_RESOLUTION_UM,
        hcr_scales=hcr_scales,
        spot_counts=spot_counts,
        hcr_metrics=hcr_metrics,
    )

    return feats, labels


In [ ]:
all_features = []
all_labels = []

for coreg_dir in coreg_dirs:
    subject_id, czstack_date = parse_coreg_dir_name(coreg_dir)
    print(f'\nExtracting training data for {subject_id}...')
    feats, labels = extract_training_examples(coreg_dir, subject_id, czstack_date)
    if feats is not None:
        feats['subject_id'] = subject_id
        all_features.append(feats)
        all_labels.append(labels)

if all_features:
    X_all = pd.concat(all_features, ignore_index=True)
    y_all = np.concatenate(all_labels)
    print(f'\nTotal training examples: {len(y_all)} ({y_all.sum()} positive, {(1-y_all).sum()} negative)')
else:
    print('WARNING: No training data extracted')
    X_all = pd.DataFrame()
    y_all = np.array([])


In [ ]:
# Leave-one-subject-out cross-validation
from sklearn.metrics import roc_auc_score

if len(all_features) >= 2:
    subjects = X_all['subject_id'].values
    unique_subjects = np.unique(subjects)
    auc_scores = []
    for held_out in unique_subjects:
        train_mask = subjects != held_out
        test_mask = subjects == held_out
        X_train = X_all[train_mask]
        y_train = y_all[train_mask]
        X_test = X_all[test_mask]
        y_test = y_all[test_mask]
        if y_test.sum() == 0 or y_train.sum() == 0:
            continue
        clf_cv = train_match_classifier(X_train, y_train)
        probs = predict_match_probability(X_test, clf_cv)
        auc = roc_auc_score(y_test, probs)
        auc_scores.append(auc)
        print(f'  Held-out {held_out}: AUC = {auc:.3f}')
    print(f'\nMean LOO-subject AUC: {np.mean(auc_scores):.3f}')


In [ ]:
# Train final classifier on all data
if len(X_all) > 0:
    final_clf = train_match_classifier(X_all, y_all)
    with open(SAVE_CLASSIFIER, 'wb') as f:
        pickle.dump(final_clf, f)
    print(f'Classifier saved to {SAVE_CLASSIFIER}')

    # Feature importance — imputer drops all-NaN columns, so filter names accordingly
    gbt = final_clf.named_steps['gbt']
    imputer = final_clf.named_steps['imputer']
    all_feat_names = np.array(final_clf.feature_names_)
    kept_mask = ~np.isnan(imputer.statistics_)
    kept_feat_names = all_feat_names[kept_mask]

    importances = gbt.feature_importances_
    print(f'Kept features after imputation: {kept_feat_names.tolist()}')
    print(f'Dropped (all-NaN): {all_feat_names[~kept_mask].tolist()}')
    assert len(kept_feat_names) == len(importances), \
        f"Mismatch: {len(kept_feat_names)} names vs {len(importances)} importances"

    imp_df = pd.DataFrame({'feature': kept_feat_names, 'importance': importances})
    imp_df = imp_df.sort_values('importance', ascending=False)
    print('\nTop feature importances:')
    print(imp_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1])
    ax.set_xlabel('Importance')
    ax.set_title('GBT Feature Importances (final classifier)')
    plt.tight_layout()
    plt.savefig('/scratch/fig_classifier_feature_importance.png', dpi=120)
    plt.show()
else:
    print('No training data - classifier not saved')